In [1]:
!pip install cake-ensemble

In [10]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, FloatSlider
from sklearn.datasets import make_blobs
from cake_ensemble import kmeans_ensemble, cake

# Pre-calculate the data and CAKE scores so the slider is fast
print("Calculating CAKE scores... please wait.\n\n")
X, _ = make_blobs(
    n_samples=4000,
    centers=[[0, 0], [5, 5], [12, 0]],
    cluster_std=2,
    random_state=42
)
labels_list, centers, _ = kmeans_ensemble(X, n_clusters=3, n_runs=10, random_state=0)
cake_scores, _, _, _ = cake(X, labels_list, method="harmonic_mean", approximation=True, centers_list = centers)

# Define the interactive plotting function
def plot_cake(threshold):
    fig, ax = plt.subplots(figsize=(10, 6))

    mask = cake_scores >= threshold

    # Plot Filtered (Uncertain) points
    ax.scatter(X[~mask, 0], X[~mask, 1], c='lightgrey', s=15, alpha=0.3, label="Filtered")

    # Plot Confident points
    scatter = ax.scatter(X[mask, 0], X[mask, 1], c=cake_scores[mask],
                         cmap='jet', s=30, label="")

    if mask.any():
        cbar = plt.colorbar(scatter)
        cbar.set_label('CAKE Score')

    ax.set_title(f"Points with CAKE Confidence ≥ {threshold:.2f}")
    ax.set_axis_off()
    plt.legend(loc='upper right')
    plt.show()
    print(f"Showing {mask.sum()} / {len(X)} points")

# Create the slider
interact(plot_cake, threshold=FloatSlider(min=0.1, max=0.99, step=0.05, value=0.0));

Calculating CAKE scores... please wait.




interactive(children=(FloatSlider(value=0.1, description='threshold', max=0.99, min=0.1, step=0.05), Output())…

## Download

In [17]:
!pip install cake-ensemble jupyter_bokeh -q

import panel as pn
import matplotlib.pyplot as plt
from google.colab import files
import numpy as np

# Initialize Panel
pn.extension(sizing_mode="stretch_width")

# Define the plot function
def get_final_plot(threshold):
    fig, ax = plt.subplots(figsize=(10, 6))
    mask = cake_scores >= threshold

    # Plot background
    ax.scatter(X[~mask, 0], X[~mask, 1], c='lightgrey', s=15, alpha=0.3, label="Filtered")

    # Plot CAKE points
    scatter = ax.scatter(X[mask, 0], X[mask, 1], c=cake_scores[mask],
                         cmap='jet', s=30)

    if mask.any():
        cbar = plt.colorbar(scatter)
        cbar.set_label('CAKE Score')

    ax.set_title(f"CAKE Confidence Threshold: {threshold:.2f}")
    ax.set_axis_off()
    plt.tight_layout()
    plt.close(fig)
    return pn.pane.Matplotlib(fig, tight=True)

slider = pn.widgets.FloatSlider(
    name='Threshold',
    start=0.1,
    end=1.0,
    step=0.05,
    value=0.1,
    width=500
)

interactive_view = pn.bind(get_final_plot, threshold=slider)
layout = pn.Column("## CAKE Interactive Explorer", slider, interactive_view)

raw_steps = np.arange(0.1, 1.0, 0.05)
clean_steps = [round(v, 2) for v in raw_steps if v <= 1.0]
steps_to_record = {slider: clean_steps}

print(f"Recording {len(clean_steps)} positions: {clean_steps}")
print("Baking plots... please wait.")

# Save and Download
layout.save('cake_smooth_explorer.html', embed=True, embed_states=steps_to_record)
files.download('cake_smooth_explorer.html')
print("Done! Open the downloaded HTML in your browser.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.1 MB/s eta 0:00:00


Recording 18 positions: [np.float64(0.1), np.float64(0.15), np.float64(0.2), np.float64(0.25), np.float64(0.3), np.float64(0.35), np.float64(0.4), np.float64(0.45), np.float64(0.5), np.float64(0.55), np.float64(0.6), np.float64(0.65), np.float64(0.7), np.float64(0.75), np.float64(0.8), np.float64(0.85), np.float64(0.9), np.float64(0.95)]
Baking plots... please wait.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Open the downloaded HTML in your browser.


## Gif

In [22]:
import imageio
import os

# Temporary folder for frames
if not os.path.exists('frames'):
    os.makedirs('frames')

print("Generating frames...")
filenames = []

# We use a smaller range for a faster/smaller GIF
for i, threshold in enumerate(np.arange(0.1, 1.0, 0.05)):
    fig, ax = plt.subplots(figsize=(8, 5))
    mask = cake_scores >= threshold
    ax.scatter(X[~mask, 0], X[~mask, 1], c='lightgrey', s=10, alpha=0.2)
    scatter = ax.scatter(X[mask, 0], X[mask, 1], c=cake_scores[mask], cmap='jet', s=20)
    ax.set_title(f"CAKE Confidence Threshold: {threshold:.2f}")
    ax.set_axis_off()

    filename = f'frames/frame_{i}.png'
    plt.savefig(filename, dpi=100)
    filenames.append(filename)
    plt.close(fig)

# Stitch into a GIF
with imageio.get_writer('demo_cake.gif', mode='I', duration=250) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

print("GIF created successfully!")
files.download('demo_cake.gif')

Generating frames...


/tmp/ipykernel_26693/955296434.py:28: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(filename)


GIF created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>